# BERT NER on CoNLL-2003 (Google Colab Ready)

**Instructions:**
1. Runtime → Change runtime type → GPU
2. Run all cells

Note: This notebook pins `datasets<3.0.0` to avoid recent breaking changes with the CoNLL-2003 loader.


In [ ]:
!pip install -q transformers "datasets<3.0.0" seqeval accelerate torch


In [ ]:
from datasets import load_dataset
dataset = load_dataset("conll2003")
label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)
print("Labels:", label_list)


In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


In [ ]:
from transformers import BertForTokenClassification
model = BertForTokenClassification.from_pretrained(
    "bert-base-cased", num_labels=num_labels
)


In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./bert-ner-conll",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"
)


In [ ]:
import numpy as np
from seqeval.metrics import f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {"f1": f1_score(true_labels, true_predictions)}


In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
from transformers import pipeline
ner_pipeline = pipeline(
    "ner", model=trainer.model, tokenizer=tokenizer, aggregation_strategy="simple"
)
text = "Sundar Pichai is the CEO of Google in California."
ner_pipeline(text)
